# Heritage Short — Stage 4: Original Score with ACE-Step 1.5

Run on Kaggle with **Accelerator: GPU T4** and **Internet: ON**.
Kaggle defaults to Python 3.13, while ACE-Step 1.5 requires Python `>=3.11,<3.13`.
This notebook uses `uv` to install an isolated Python 3.12 environment cleanly.

In [ ]:
!rm -rf /kaggle/working/ACE-Step-1.5
!git clone https://github.com/ACE-Step/ACE-Step-1.5.git /kaggle/working/ACE-Step-1.5
%cd /kaggle/working/ACE-Step-1.5
!pip install -q uv
!uv python install 3.12
!uv sync --python 3.12

In [ ]:
%%writefile /kaggle/working/ACE-Step-1.5/generate_candidates.py
import os
os.environ['MPLBACKEND'] = 'Agg'

import json
import shutil
import time
from pathlib import Path

from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music
from acestep.model_downloader import ensure_main_model, ensure_lm_model

CAPTION = (
    'Ethiopian traditional acoustic instrumental folk underscore, warm and nostalgic, '
    'gentle krar lyre melody, subtle masinko fiddle, soft washint flute, sparse unhurried 72 BPM tempo, '
    'cinematic film score leaving open space for spoken voiceover, subtle tension in the middle '
    'resolving into peaceful warmth, strictly instrumental no vocals'
)
LYRICS = '[Instrumental]'
DURATION = 80.0
BPM = 72
KEYSCALE = 'A minor'
SEEDS = [42, 123, 777]
OUT_DIR = '/kaggle/working/music_candidates'
os.makedirs(OUT_DIR, exist_ok=True)

ckpt_dir = Path('/kaggle/working/ACE-Step-1.5/checkpoints')
ckpt_dir.mkdir(parents=True, exist_ok=True)
print('Ensuring checkpoints downloaded...')
ensure_main_model(checkpoints_dir=ckpt_dir)
ensure_lm_model(model_name='acestep-5Hz-lm-0.6B', checkpoints_dir=ckpt_dir)

print('Initializing AceStepHandler (DiT)...')
dit_handler = AceStepHandler()
init_status, enable_gen = dit_handler.initialize_service(
    project_root='/kaggle/working/ACE-Step-1.5',
    config_path='acestep-v15-turbo',
    device='cuda',
)
if not enable_gen:
    raise RuntimeError(f'DiT init failed: {init_status}')

print('Initializing LLMHandler...')
llm_handler = LLMHandler()
lm_status, lm_ok = llm_handler.initialize(
    checkpoint_dir=str(ckpt_dir),
    lm_model_path='acestep-5Hz-lm-0.6B',
    backend='pt',
    device='cuda',
)
if not lm_ok:
    raise RuntimeError(f'LM init failed: {lm_status}')

logs = []
for seed in SEEDS:
    t0 = time.time()
    params = GenerationParams(
        task_type='text2music',
        caption=CAPTION,
        lyrics=LYRICS,
        bpm=BPM,
        keyscale=KEYSCALE,
        duration=DURATION,
    )
    config = GenerationConfig(batch_size=1, audio_format='wav', use_random_seed=False, seeds=[seed])
    seed_dir = os.path.join('/kaggle/working/ACE-Step-1.5/_raw', f'seed_{seed}')
    os.makedirs(seed_dir, exist_ok=True)
    result = generate_music(
        dit_handler=dit_handler,
        llm_handler=llm_handler,
        params=params,
        config=config,
        save_dir=seed_dir,
    )
    elapsed = round(time.time() - t0, 2)
    saved_path = None
    if result.success and result.audios:
        src = result.audios[0]['path']
        dst = os.path.join(OUT_DIR, f'score_seed_{seed}.wav')
        shutil.copy2(src, dst)
        saved_path = dst
        print(f'OK seed={seed} -> {dst} ({elapsed}s)')
    else:
        print(f'FAIL seed={seed}: {getattr(result, "error", "unknown")}')
    logs.append({
        'stage': 'music',
        'model': 'ACE-Step-1.5',
        'seed': seed,
        'bpm': BPM,
        'key': KEYSCALE,
        'duration_seconds': DURATION,
        'latency_seconds': elapsed,
        'output_file': saved_path,
        'status': 'success' if saved_path else 'error',
    })

for log_path in ['/kaggle/working/music_runs.jsonl', '/kaggle/working/00_music_runs.jsonl']:
    with open(log_path, 'w') as f:
        for row in logs:
            f.write(json.dumps(row) + '\n')
shutil.make_archive('/kaggle/working/00_music_candidates', 'zip', '/kaggle/working', 'music_candidates')
print('Done! Outputs in /kaggle/working/music_candidates/ and /kaggle/working/00_music_candidates.zip')


In [ ]:
import os, pathlib, subprocess
env = os.environ.copy()
env['MPLBACKEND'] = 'Agg'
subprocess.run(
    ['uv', 'run', '--python', '3.12', 'python', '/kaggle/working/ACE-Step-1.5/generate_candidates.py'],
    cwd='/kaggle/working/ACE-Step-1.5',
    env=env,
    check=True,
)
for p in sorted(pathlib.Path('/kaggle/working/music_candidates').glob('*.wav')):
    print(p, round(p.stat().st_size / 1024, 1), 'KB')
print(pathlib.Path('/kaggle/working/00_music_runs.jsonl').read_text())